In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import csv
import os

# Configuration
INPUT_FILE = '/Users/vconklin24/Desktop/data/book_ids/goodreads_ids_temp.csv'
OUTPUT_FILE = 'data/raw/goodreads_current.csv'
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'

def scrape_book_stats(url):
    headers = {'User-Agent': USER_AGENT}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Dictionary to hold our scraped data
        stats = {
            'avg_rating': None,
            'rating_count': None,
            'review_count': None,
            'want_to_read': None,
            'dist_5_star': None,
            'dist_4_star': None,
            'dist_3_star': None,
            'dist_2_star': None,
            'dist_1_star': None
        }

        # 1. Average Rating
        avg_rating_tag = soup.find("div", {"class": "RatingStatistics__rating"})
        if avg_rating_tag:
            stats['avg_rating'] = avg_rating_tag.text.strip()

        # 2. Ratings and Reviews Count
        meta_stats = soup.find("div", {"class": "RatingStatistics__meta"})
        if meta_stats:
            # Usually format: "1,234,567 ratings 12,345 reviews"
            text = meta_stats.get_text(separator="|")
            parts = text.split("|")
            stats['rating_count'] = parts[0].replace('ratings', '').replace(',', '').strip()
            if len(parts) > 1:
                stats['review_count'] = parts[-1].replace('reviews', '').replace(',', '').strip()

        # 3. Rating Distribution (Percentages)
        dist_tags = soup.find_all("div", {"class": "RatingsHistogram__labelTotal"})
        # These appear in order 5, 4, 3, 2, 1
        if len(dist_tags) >= 5:
            stats['dist_5_star'] = dist_tags[0].text.strip()
            stats['dist_4_star'] = dist_tags[1].text.strip()
            stats['dist_3_star'] = dist_tags[2].text.strip()
            stats['dist_2_star'] = dist_tags[3].text.strip()
            stats['dist_1_star'] = dist_tags[4].text.strip()

        return stats

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

def main():
    # Ensure directory exists
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    
    # Load your current IDs
    df = pd.read_csv(INPUT_FILE)
    
    # Filter only for rows where a URL was found
    df_to_scrape = df[df['found'] == True].copy()
    
    results = []

    print(f"Starting scrape for {len(df_to_scrape)} books...")

    for index, row in df_to_scrape.iterrows():
        print(f"Processing: {row['book_title']}...")
        
        book_stats = scrape_book_stats(row['goodreads_url'])
        
        if book_stats:
            # Combine original info with new stats
            combined_data = {**row.to_dict(), **book_stats}
            results.append(combined_data)
        
        # Respectful delay: 3-5 seconds
        wait_time = random.uniform(3, 5)
        time.sleep(wait_time)

    # Save to CSV
    final_df = pd.DataFrame(results)
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Done! Data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Starting scrape for 256 books...
Processing: Nineteen Minutes...
Processing: The Perks of Being a Wallflower...
Processing: Beloved...
Processing: Crank...
Processing: Fallout...
Processing: Glass...
Processing: Identical...
Processing: It's Your World - If You Don't Like It, Change It: Activism for Teenagers...
Processing: Looking for Alaska...
Processing: Melissa (George)...
Processing: Nineteen Minutes...
Processing: Redwood and Ponytail...
Processing: Smoke...
Processing: The Bluest Eye...
Processing: The Hate U Give...
Processing: The Kite Runner...
Processing: The Perks of Being a Wallflower...
Processing: Thirteen Reasons Why...
Processing: You Should See Me in a Crown...
Processing: Ana on the Edge...
Processing: The Ship We Built...
Processing: You Don’t Know Everything, Jilly P!...
Processing: A Thousand Splendid Suns...
Processing: Blood of my Blood...
Processing: Into the Garden...
Processing: The Human Stain...
Processing: All Boys Aren't Blue...
Processing: Damsel...
Proc